# LangChain Messages 实战：对接 DeepSeek API

与 `agent_messages_demo.py`（纯演示消息对象）不同，这个 notebook 真正调用了 **DeepSeek API**。

运行前请确保：
1. 项目根目录有 `.env` 文件（含 DeepSeek API Key）
2. 已执行 `uv sync` 安装依赖
3. notebook 在项目根目录运行（`.env` 在同一目录）

In [2]:
# ============================================================
# 公共导入（所有 Demo 共享）
# ============================================================
import os, json, sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# 加载 .env（notebook 工作目录就是项目根目录时这样写）
env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

# 消息类型
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

# 模型入口
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="deepseek-chat",
    temperature=0.7,
    api_key=os.getenv("Deepseek_API_KEY"),
    model_provider="deepseek",
)

def separator(title: str) -> None:
    """辅助函数"""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

print("✅ 所有依赖导入成功，模型已就绪")
print(f"   模型: deepseek-chat (provider=deepseek)")

✅ 所有依赖导入成功，模型已就绪
   模型: deepseek-chat (provider=deepseek)


## 01 Simple Chat

发送一条 SystemMessage + 一条 HumanMessage，拿到 AIMessage。

关键概念：
  model.invoke(messages) → 返回一条 AIMessage
  invoke 是 LangChain 最基础的调用方式，给定输入，返回完整输出


In [ ]:
separator("Demo 01 simple chat")
separator("Demo 1：一次性对话（invoke）")
messages = [
        # SystemMessage 在最前面，定义 AI 的行为
        SystemMessage(content="你是一个 Linux 专家，回答简洁专业，尽量用命令示例说明。"),
        # HumanMessage 是用户的实际问题
        HumanMessage(content="如何查看当前目录下占用空间最大的 5 个文件？"),
    ]
response: AIMessage = model.invoke(messages)
print(f"🤖 DeepSeek 的回复：\n")
print(response.content)
print()
print(f"📊 元数据：")
print(f"   model:       {response.response_metadata.get('model_name', 'N/A')}")
print(f"   finish_reason: {response.response_metadata.get('finish_reason', 'N/A')}")
print(f"   输入 token:    {response.usage_metadata.get('input_tokens', 'N/A')}")
print(f"   输出 token:    {response.usage_metadata.get('output_tokens', 'N/A')}")

## 02 Multi Turn

多轮对话的核心：把每轮的 AIMessage 追加到消息列表，再传给下一次 invoke。

关键概念：
  对话历史 = messages 列表
  每次调用前把新的 HumanMessage 追加进去
  每次调用后把返回的 AIMessage 追加进去
  这样 AI 就"记得"之前说了什么


In [4]:
separator("Demo 02 multi turn")
separator("Demo 2：多轮对话（上下文管理）")
messages = [
        SystemMessage(content="你是一个 Python 编程老师，回答要循序渐进。"),
    ]
messages.append(HumanMessage(content="Python 中列表和元组有什么区别？"))
response = model.invoke(messages)
messages.append(response)
print("第 1 轮：")
print(f"  👤: 列表和元组有什么区别？")
print(f"  🤖: {response.content[:80]}...")
print()
messages.append(HumanMessage(content="那什么时候应该用它而不是列表？"))
response = model.invoke(messages)
messages.append(response)
print("第 2 轮（AI 记得上文）：")
print(f"  👤: 那什么时候应该用它而不是列表？")
print(f"  🤖: {response.content[:120]}...")
print()
print(f"📊 当前对话历史共 {len(messages)} 条消息")
for i, m in enumerate(messages):
        role = {"system": "⚙️", "human": "👤", "ai": "🤖"}.get(m.type, "  ")
        c = m.content[:60].replace("\n", " ") + "..."
        print(f"  [{i}] {role} {c}")


  Demo 02 multi turn

  Demo 2：多轮对话（上下文管理）
第 1 轮：
  👤: 列表和元组有什么区别？
  🤖: 当然可以。让我们从最基础的区别开始，一步步深入理解。

## 第一步：最核心的区别（一句话总结）

**列表(list)可以修改，元组(tuple)不可修改。*...

第 2 轮（AI 记得上文）：
  👤: 那什么时候应该用它而不是列表？
  🤖: 这个问题问得好！让我从实际开发场景出发，详细说明什么时候应该选择元组而不是列表。

## 场景一：数据不应该被意外修改

### 典型例子：配置信息
```python
# 错误做法 - 用列表存储不应该修改的数据
colors = ["r...

📊 当前对话历史共 5 条消息
  [0] ⚙️ 你是一个 Python 编程老师，回答要循序渐进。...
  [1] 👤 Python 中列表和元组有什么区别？...
  [2] 🤖 当然可以。让我们从最基础的区别开始，一步步深入理解。  ## 第一步：最核心的区别（一句话总结）  **列表(list)...
  [3] 👤 那什么时候应该用它而不是列表？...
  [4] 🤖 这个问题问得好！让我从实际开发场景出发，详细说明什么时候应该选择元组而不是列表。  ## 场景一：数据不应该被意外修改 ...


## 03 Tool Binding

演示最关键的功能：给模型绑定工具，让它能调用外部函数。

流程：
  1. 定义工具的 schema（名称、描述、参数格式）
  2. 用 model.bind_tools() 绑定工具
  3. 发送用户消息
  4. 模型返回 AIMessage（含 tool_calls）
  5. 手动执行工具，把结果用 ToolMessage 返回
  6. 再次调用模型，让它基于工具结果给出最终回答

这就是你之前 agent.py 和 get_weather.py 的核心逻辑，
只不过现在用 LangChain 的 Messages 来管理整个流程。


In [5]:
# ============================================================
# Demo 3：工具调用 — 直接用 openai SDK（和 agent.py 一样）
# ============================================================
# ⚠️ LangChain 的 invoke(tools=...) / bind_tools() 和 DeepSeek 有兼容问题
#    所以这里回归最原始的做法：用 openai SDK 手动处理 tool_calls

from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam, ChatCompletionToolParam

TOOLS: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取指定时区的当前时间",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "时区名称，如 Asia/Shanghai、America/New_York",
                    }
                },
                "required": ["timezone"],
            },
        },
    },
]

# 用 openai SDK 穿 DeepSeek（和 agent.py 一模一样）
client = OpenAI(
    api_key=os.getenv("Deepseek_API_KEY"),
    base_url="https://api.deepseek.com/v1",
)

# 构建 langchain messages，后面转换给 openai sdk 用
from langchain_core.messages import convert_to_openai_messages

lc_messages = [
    SystemMessage(content="你是一个时间助手。用户问时间就调 get_current_time 工具。"),
    HumanMessage(content="现在北京时间几点？"),
]

# ---- 第 1 步：把 langchain messages 转成 OpenAI 格式，发给 DeepSeek ----
openai_msgs = convert_to_openai_messages(lc_messages)
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=openai_msgs,
    tools=TOOLS,
    tool_choice="auto",
    temperature=0.7,
)

assistant_msg = response.choices[0].message
print("第 1 次调用 —— 模型决定调用工具：")
print(f"  content:     '{assistant_msg.content}'")
print(f"  tool_calls:  {len(assistant_msg.tool_calls or [])} 个")
for tc in (assistant_msg.tool_calls or []):
    print(f"    → 工具名: {tc.function.name}")
    print(f"    → 参数:   {tc.function.arguments}")
    print(f"    → ID:     {tc.id}")

# ---- 第 2 步：执行工具，把结果追加进去 ----
from datetime import datetime
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 把 assistant_msg 和 tool_result 追加进来（用 openai 原生 dict 格式）
openai_msgs.append(assistant_msg.model_dump())

for tc in (assistant_msg.tool_calls or []):
    openai_msgs.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": json.dumps({
            "timezone": json.loads(tc.function.arguments)["timezone"],
            "current_time": current_time,
            "timezone_offset": "+08:00",
        }, ensure_ascii=False),
    })
    print(f"\n工具 get_current_time 执行结果：{current_time}")

# ---- 第 3 步：第二次调用，模型给出最终回答 ----
response2 = client.chat.completions.create(
    model="deepseek-chat",
    messages=openai_msgs,
    tools=TOOLS,
    temperature=0.7,
)

final = response2.choices[0].message.content
print(f"\n第 2 次调用 —— 基于工具结果的最终回答：")
print(f"  🤖: {final}")

# ---- 回顾整个对话流程 ----
print(f"\n📊 整个对话的消息流转（原生 OpenAI 格式）：")
for i, m in enumerate(openai_msgs):
    role = m.get("role", "?")
    content = str(m.get("content", ""))[:60].replace("\n", " ")
    if m.get("tool_calls"):
        content = f"[调用工具: {[tc['function']['name'] for tc in m['tool_calls']]}])]"
    print(f"  [{i}] {role:10s} {content}...")



  Demo 03 tool binding

  Demo 3：工具调用（Tool Call / Function Calling）
第一次调用 —— 模型返回 tool_calls：
  content:     '' ← 为空，因为模型决定调工具
  tool_calls:  1 个
    → 工具名: get_current_time
    → 参数:   {'timezone': 'Asia/Shanghai'}
    → ID:     call_00_cYDSjGdloPFzthQD4EYK5512

工具 get_current_time 执行结果：2026-07-05 13:33:32

第二次调用 —— 基于工具结果的最终回答：
  🤖: 现在是北京时间 **2026年7月5日 13:33:32**（东八区，UTC+8）。😊

📊 整个对话的消息流转：
  [0] ⚙️  system: 你是一个时间助手。用户问时间就调 get_current_time 工具。...
  [1] 👤  human: 现在北京时间几点？
  [2] 🤖  ai:     [调用工具: ['get_current_time']]
  [3] 🔧  tool:   {"timezone": "Asia/Shanghai", "current_time": "202...
  [4] 🤖  ai:     现在是北京时间 **2026年7月5日 13:33:32**（东八区，UTC+8）。😊...


## 04 Streaming

流式输出：一句话一句话地返回，而不是等全写完再返回。

关键概念：
  model.stream(messages) → 返回一个迭代器，每个迭代是一段增量文本
  用户体验更好（像 ChatGPT 那样逐字输出）

注意：
  - stream 每次只返回 delta（增量），不是完整内容
  - token_usage 信息在最后一个 chunk 里
  - 有 tool_calls 时，流式返回可能不完整，建议用 invoke


In [6]:
separator("Demo 04 streaming")
separator("Demo 4：流式输出（stream）")
messages = [
        SystemMessage(content="你是一个简洁的助手，回答问题不超过 3 句话。"),
        HumanMessage(content="解释一下什么是递归，给一个简短的 Python 例子"),
    ]
print("🤖 DeepSeek 流式回复（逐字输出效果）：\n")
full_content = ""
for chunk in model.stream(messages):
        # chunk 是 AIMessageChunk，.content 是这一小段增量文本
        if chunk.content:
            print(chunk.content, end="", flush=True)
            full_content += str(chunk.content)
print(f"\n\n📊 完整回复共 {len(full_content)} 个字符")


  Demo 04 streaming

  Demo 4：流式输出（stream）
🤖 DeepSeek 流式回复（逐字输出效果）：

递归是函数调用自身的编程技巧，通常用于分解问题为更小的同类子问题。例如计算阶乘：

```python
def factorial(n):
    return 1 if n == 0 else n * factorial(n-1)
```

📊 完整回复共 120 个字符


## 05 System Message Effect

对比 SystemMessage 的实际效果：
  同样的问题，有/没有 SystemMessage 给出完全不同的风格

这展示了 SystemMessage 的核心价值：
  它不是装饰品，而是切实控制 AI 的输出风格和范围


In [7]:
separator("Demo 05 system message effect")
separator("Demo 5：SystemMessage 的实际影响对比")
question = "解释一下什么是闭包"
messages_a = [HumanMessage(content=question)]
response_a = model.invoke(messages_a)
messages_b = [
        SystemMessage(content="用幼儿园小朋友都能听懂的大白话解释，不要用任何术语，50 字以内。"),
        HumanMessage(content=question),
    ]
response_b = model.invoke(messages_b)
print("同一个问题的两种回答风格：\n")
print(f"❌ 无 SystemMessage（默认风格）：")
print(f"   {response_a.content[:200]}...")
print()
print(f"✅ 有 SystemMessage（大白话风格）：")
print(f"   {response_b.content}")
print()
print("结论：SystemMessage 对 AI 行为的影响是非常显著的。")


  Demo 05 system message effect

  Demo 5：SystemMessage 的实际影响对比
同一个问题的两种回答风格：

❌ 无 SystemMessage（默认风格）：
   这是一个非常核心且经典的编程概念。简单来说，**闭包**就是一个函数“记住”并能够访问它**词法作用域**之外的变量，即使这个函数是在其词法作用域之外被执行的。

听起来有点绕？我们拆开来看。

### 核心三要素

一个闭包通常由三部分组成：

1.  **一个外部函数**：它定义了一个内部函数，并可能返回这个内部函数。
2.  **一个内部函数**：它“引用”了外部函数作用域中的变量（参数或局...

✅ 有 SystemMessage（大白话风格）：
   闭包就像一个会记事的盒子，里面装着之前的东西，以后还能拿出来用。

结论：SystemMessage 对 AI 行为的影响是非常显著的。
